In [ ]:
"""
Stripe Documentation Scraper
Scrapes Stripe API docs, error codes, and troubleshooting guides
"""

import json
import time
from pathlib import Path
from typing import Dict, List

import bs4
import requests
from bs4 import BeautifulSoup
from langchain_classic.document_loaders import WebBaseLoader


class StripeDocScraper:
    def __init__(self, output_dir: str = "data/stripe_docs"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def _categorize_doc(self, url: str) -> str:
        """Categorize document based on URL"""
        if "error" in url.lower():
            return "error_code"
        elif "webhook" in url.lower():
            return "webhook"
        elif "authentication" in url.lower() or "api-key" in url.lower():
            return "authentication"
        elif "payment" in url.lower():
            return "payment"
        else:
            return "general"

    def scrape_stripe_docs(self) -> List[Dict]:
        """Scrape key Stripe documentation pages using LangChain WebBaseLoader"""

        # Core Stripe documentation URLs
        urls = [
            # Authentication
            "https://stripe.com/docs/keys",
            "https://stripe.com/docs/api/authentication",
            # Error handling
            "https://stripe.com/docs/api/errors",
            "https://stripe.com/docs/error-codes",
            # Webhooks
            "https://stripe.com/docs/webhooks",
            "https://stripe.com/docs/webhooks/signatures",
            "https://stripe.com/docs/webhooks/test",
            # Payments
            "https://stripe.com/docs/payments",
            "https://stripe.com/docs/payments/accept-a-payment",
            "https://stripe.com/docs/payments/payment-intents",
            # API
            "https://stripe.com/docs/api/idempotent_requests",
            "https://stripe.com/docs/api/versioning",
            "https://stripe.com/docs/api/rate_limits",
            # Common issues
            "https://stripe.com/docs/testing",
            "https://stripe.com/docs/disputes",
        ]

        print(f"Loading {len(urls)} Stripe documentation pages...")

        # Optional: Use bs4.SoupStrainer to only parse relevant content
        # This filters out navigation, footers, ads, etc.
        bs4_strainer = bs4.SoupStrainer(
            class_=("content", "docs-content", "markdown", "api-reference")
        )

        try:
            # Load all documents at once
            # WebBaseLoader handles requests, parsing, and error handling
            loader = WebBaseLoader(
                web_paths=urls,
                bs_kwargs={"parse_only": bs4_strainer},
                # Add headers to avoid being blocked
                requests_kwargs={
                    "headers": {
                        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
                    }
                },
            )

            # This does the actual scraping
            langchain_docs = loader.load()

            print(f"✅ Successfully loaded {len(langchain_docs)} documents")

            # Convert LangChain Documents to our format
            formatted_docs = []
            for doc in langchain_docs:
                # Extract metadata
                source_url = doc.metadata.get("source", "unknown")

                # Only keep documents with substantial content
                if len(doc.page_content.strip()) > 100:
                    formatted_docs.append(
                        {
                            "url": source_url,
                            "title": doc.metadata.get(
                                "title", self._extract_title_from_url(source_url)
                            ),
                            "content": doc.page_content.strip(),
                            "doc_type": self._categorize_doc(source_url),
                        }
                    )

        except Exception as e:
            print(f"⚠️ Error during scraping: {e}")
            print("Falling back to synthetic documents...")
            formatted_docs = []

        return formatted_docs

    def _extract_title_from_url(self, url: str) -> str:
        """Extract a title from the URL if metadata doesn't have one"""
        # Extract last part of URL path
        parts = url.rstrip("/").split("/")
        if parts:
            title = parts[-1].replace("-", " ").replace("_", " ").title()
            return title
        return "Untitled"

    def save_docs(self, docs: List[Dict]):
        """Save documents to JSON file"""
        output_file = self.output_dir / "stripe_docs.json"
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(docs, f, indent=2, ensure_ascii=False)
        print(f"💾 Saved {len(docs)} documents to {output_file}")

    def run(self):
        """Main scraping workflow"""
        print("=" * 60)
        print("Starting Stripe documentation scraping...")
        print("=" * 60)

        # Try to scrape real docs using LangChain WebBaseLoader
        scraped_docs = self.scrape_stripe_docs()

        print(f"   - Scraped from web: {len(scraped_docs)}")

        # Save combined docs
        self.save_docs(scraped_docs)

        # Print breakdown by type
        doc_types = {}
        for doc in scraped_docs:
            doc_type = doc.get("doc_type", "unknown")
            doc_types[doc_type] = doc_types.get(doc_type, 0) + 1

        print("\n📋 Document breakdown by type:")
        for doc_type, count in sorted(doc_types.items()):
            print(f"   {doc_type}: {count}")

        print("=" * 60)

        return scraped_docs



scraper = StripeDocScraper()
docs = scraper.run()

# Print sample of first document
if docs:
    print("\n📄 Sample from first document:")
    print(f"Title: {docs[0]['title']}")
    print(f"URL: {docs[0]['url']}")
    print(f"Type: {docs[0]['doc_type']}")
    print(f"Content preview: {docs[0]['content'][:200]}...")


In [ ]:
import requests

def check_urls(url_list):
    """
    Checks the status of a list of URLs using HEAD requests.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    print("Checking URLs...")
    for url in url_list:
        try:
            # Use a HEAD request for efficiency, timeout after 5 seconds
            response = requests.head(url, headers=headers, timeout=5, allow_redirects=True)
            
            # Check if the status code is in the 2xx (successful) or 3xx (redirection) range
            if response.ok:
                print(f"✅ SUCCESS: {url} (Status: {response.status_code})")
            else:
                print(f"❌ FAILED:  {url} (Status: {response.status_code})")

        except requests.exceptions.RequestException as e:
            # Handle connection errors, timeouts, etc.
            print(f"‼️ ERROR:   {url} (Could not connect: {e.__class__.__name__})")

# The list of URLs to check
urls_to_test = [
    # Authentication
    "https://stripe.com/docs/keys",
    "https://stripe.com/docs/api/authentication",
    
    # Error handling
    "https://stripe.com/docs/api/errors",
    "https://stripe.com/docs/error-codes",
    
    # Webhooks
    "https://stripe.com/docs/webhooks",
    "https://stripe.com/docs/webhooks/signatures",
    "https://stripe.com/docs/webhooks/test",
    
    # Payments
    "https://stripe.com/docs/payments",
    "https://stripe.com/docs/payments/accept-a-payment",
    "https://stripe.com/docs/payments/payment-intents",
    
    # API
    "https://stripe.com/docs/api/idempotent_requests",
    "https://stripe.com/docs/api/versioning",
    "https://docs.stripe.com/rate-limits",
    
    # Common issues
    "https://stripe.com/docs/testing",
    "https://stripe.com/docs/disputes",
]

# Run the check
if __name__ == "__main__":
    check_urls(urls_to_test)

Checking URLs...
✅ SUCCESS: https://stripe.com/docs/keys (Status: 200)
✅ SUCCESS: https://stripe.com/docs/api/authentication (Status: 200)
✅ SUCCESS: https://stripe.com/docs/api/errors (Status: 200)
✅ SUCCESS: https://stripe.com/docs/error-codes (Status: 200)
✅ SUCCESS: https://stripe.com/docs/webhooks (Status: 200)
✅ SUCCESS: https://stripe.com/docs/webhooks/signatures (Status: 200)
✅ SUCCESS: https://stripe.com/docs/webhooks/test (Status: 200)
✅ SUCCESS: https://stripe.com/docs/payments (Status: 200)
✅ SUCCESS: https://stripe.com/docs/payments/accept-a-payment (Status: 200)
✅ SUCCESS: https://stripe.com/docs/payments/payment-intents (Status: 200)
✅ SUCCESS: https://stripe.com/docs/api/idempotent_requests (Status: 200)
‼️ ERROR:   https://stripe.com/docs/api/versioning (Could not connect: ReadTimeout)
❌ FAILED:  https://stripe.com/docs/api/rate_limits (Status: 404)
‼️ ERROR:   https://stripe.com/docs/testing (Could not connect: ReadTimeout)
‼️ ERROR:   https://stripe.com/docs/disputes

In [ ]:
# Test script

from agents.confidence import ConfidenceCalculator
from backend.agents.generator import LLMGenerator
from scripts.vector_store import VectorStoreManager

# Initialize components
vs_manager = VectorStoreManager()  # Uses NVIDIA or fallback
llm_gen = LLMGenerator()
conf_calc = ConfidenceCalculator()  # No embedding function yet

# Create workflow (auto-links embeddings)
workflow = TicketWorkflow(vs_manager, llm_gen, conf_calc)

# Now conf_calc.embedding_function is set!
print(f"Embedding provider: {vs_manager.embedding_provider}")
print(f"Confidence calculator has embeddings: {conf_calc.embedding_function is not None}")

# Process a ticket
test_ticket = {
    'ticket_id': 'TEST-001',
    'subject': 'API key authentication error',
    'description': 'Getting 401 errors after regenerating my API key',
    'priority': 'medium'
}

result = workflow.process_ticket(test_ticket)